In [30]:
import pandas as pd
df=pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\commercial_clean.csv')

In [31]:
df.iloc[0]

ddl_case_id               01-03-02-207401000012010
year                                          2010
state_code                                       1
dist_code                                        3
court_no                                         2
cino                              MHJG040004222010
judge_position         district and sessions court
female_defendant                          1 female
female_petitioner                           0 male
female_adv_def                                   0
female_adv_pet                               -9998
type_name                                   4784.0
purpose_name                                4511.0
disp_name                                       25
date_of_filing                          2010-02-11
date_of_decision                        2011-03-30
date_first_list                         2010-02-26
date_last_list                          2011-03-16
date_next_list                          2011-03-30
act                            

In [32]:
cols_to_keep = [
    'ddl_case_id', 'cino', 'state_code', 'dist_code', 'court_no', 
    'judge_position', 'type_name', 'act', 'section',
    'date_of_filing', 'date_of_decision',
    'resolution_days', 'resolution_bucket'
]

df_model = df[cols_to_keep].copy()
print(df_model.shape)
print(df_model.isnull().sum())

(2203465, 13)
ddl_case_id             0
cino                    0
state_code              0
dist_code               0
court_no                0
judge_position          0
type_name               0
act                  1172
section                 0
date_of_filing          0
date_of_decision        0
resolution_days         0
resolution_bucket       0
dtype: int64


In [33]:
print(f"Resolved cases only: {df_model['date_of_decision'].notna().all()}")
print(f"Resolution bucket distribution:")
print(df_model['resolution_bucket'].value_counts().sort_index())

Resolved cases only: True
Resolution bucket distribution:
resolution_bucket
0_under_6months      852085
1_six_to_24months    800170
2_over_2years        551210
Name: count, dtype: int64


In [34]:
df_model['date_of_filing'] = pd.to_datetime(df_model['date_of_filing'])

df_model['filing_year'] = df_model['date_of_filing'].dt.year
df_model['filing_month'] = df_model['date_of_filing'].dt.month
df_model['filing_quarter'] = df_model['date_of_filing'].dt.quarter
df_model['filing_dayofweek'] = df_model['date_of_filing'].dt.dayofweek

print(df_model[['filing_year','filing_month','filing_quarter','filing_dayofweek']].head(10))
print(f"\nFiling year distribution:")
print(df_model['filing_year'].value_counts().sort_index())

   filing_year  filing_month  filing_quarter  filing_dayofweek
0         2010             2               1                 3
1         2010            12               4                 4
2         2010             3               1                 1
3         2010            12               4                 5
4         2010             6               2                 3
5         2010             6               2                 1
6         2010             6               2                 3
7         2010             7               3                 4
8         2010             7               3                 2
9         2010             7               3                 0

Filing year distribution:
filing_year
2010    102801
2011    130551
2012    177791
2013    205465
2014    321991
2015    352487
2016    313982
2017    317501
2018    280896
Name: count, dtype: int64


In [35]:
dow_stats = df_model.groupby('filing_dayofweek')['resolution_days'].median().round(0)
dow_stats.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
print("Median resolution by day of filing:")
print(dow_stats)

# Does month matter?
month_stats = df_model.groupby('filing_month')['resolution_days'].median().round(0)
print("\nMedian resolution by month of filing:")
print(month_stats)

# Does year matter?
year_stats = df_model.groupby('filing_year')['resolution_days'].median().round(0)
print("\nMedian resolution by filing year:")
print(year_stats)

Median resolution by day of filing:
Mon    312.0
Tue    300.0
Wed    302.0
Thu    286.0
Fri    278.0
Sat    273.0
Sun    146.0
Name: resolution_days, dtype: float64

Median resolution by month of filing:
filing_month
1     327.0
2     306.0
3     335.0
4     303.0
5     293.0
6     340.0
7     314.0
8     331.0
9     302.0
10    246.0
11    218.0
12    237.0
Name: resolution_days, dtype: float64

Median resolution by filing year:
filing_year
2010    846.0
2011    917.0
2012    730.0
2013    558.0
2014    389.0
2015    388.0
2016    262.0
2017    133.0
2018     45.0
Name: resolution_days, dtype: float64


In [36]:
## very importnat 2017 and 2018 less resolution times , survival bias, becasuse the cases not resolved are deleted becasue of no date of descision
df_model = df_model[df_model['filing_year'] <= 2015].copy()

print(f"Rows after dropping 2016-2018: {len(df_model):,}")
print(f"\nFiling year distribution:")
print(df_model['filing_year'].value_counts().sort_index())
print(f"\nTarget distribution:")
print(df_model['resolution_bucket'].value_counts().sort_index())

Rows after dropping 2016-2018: 1,291,086

Filing year distribution:
filing_year
2010    102801
2011    130551
2012    177791
2013    205465
2014    321991
2015    352487
Name: count, dtype: int64

Target distribution:
resolution_bucket
0_under_6months      312494
1_six_to_24months    477213
2_over_2years        501379
Name: count, dtype: int64


In [37]:
df_model.head()

,ddl_case_id,cino,state_code,dist_code,court_no,judge_position,type_name,act,section,date_of_filing,date_of_decision,resolution_days,resolution_bucket,filing_year,filing_month,filing_quarter,filing_dayofweek
0,01-03-02-207401000012010,MHJG040004222010,1,3,2,district and sessions court,4784.0,7276.0,1345422.0,2010-02-11,2011-03-30,412,1_six_to_24months,2010,2,1,3
1,01-03-02-207401000012011,MHJG040039232010,1,3,2,district and sessions court,4784.0,16320.0,635313.0,2010-12-24,2012-07-11,565,1_six_to_24months,2010,12,4,4
2,01-03-02-207401000022010,MHJG040006132010,1,3,2,district and sessions court,4784.0,16320.0,635313.0,2010-03-02,2012-06-25,846,2_over_2years,2010,3,1,1
3,01-03-02-207401000022011,MHJG040036672010,1,3,2,district and sessions court,4784.0,16320.0,643810.0,2010-12-04,2012-12-17,744,2_over_2years,2010,12,4,5
4,01-03-02-207401000032010,MHJG040016072010,1,3,2,district and sessions court,4784.0,16320.0,678419.0,2010-06-10,2011-01-03,207,1_six_to_24months,2010,6,2,3


In [38]:
# Quarter vs month signal
quarter_stats = df_model.groupby('filing_quarter')['resolution_days'].median().round(0)
print("Median resolution by quarter:")
print(quarter_stats)

month_stats = df_model.groupby('filing_month')['resolution_days'].median().round(0)
print("\nMedian resolution by month:")
print(month_stats)

year_stats = df_model.groupby('filing_year')['resolution_days'].median().round(0)
print("\nMedian resolution by year (2010-2015 only):")
print(year_stats)


# Quarter — Keep, drop month. Quarter shows a strong pattern — Q4 (385 days) vs Q1 (590 days) is a 53% difference.
# Year — Keep it. 2010 to 2015 range is 846 to 388 days — that's a 54% difference with no survivorship bias. This is real — India introduced fast track commercial courts progressively, backlogs shifted

Median resolution by quarter:
filing_quarter
1    590.0
2    567.0
3    547.0
4    385.0
Name: resolution_days, dtype: float64

Median resolution by month:
filing_month
1     596.0
2     581.0
3     593.0
4     588.0
5     553.0
6     552.0
7     560.0
8     552.0
9     529.0
10    402.0
11    363.0
12    398.0
Name: resolution_days, dtype: float64

Median resolution by year (2010-2015 only):
filing_year
2010    846.0
2011    917.0
2012    730.0
2013    558.0
2014    389.0
2015    388.0
Name: resolution_days, dtype: float64


In [39]:
df_model.drop(columns=['filing_month', 'filing_dayofweek'], inplace=True)

In [40]:
df_model['judge_position'].value_counts()

judge_position
munsiff first class court                  150784
principal civil judge                      127553
civil judge senior division                125092
district and sessions court                112132
judicial magistrate court                  106555
                                            ...  
magistrate court miryalaguda                    1
co-operative appellate court aurangabad         1
gram nyayalaya                                  1
magistrate courts                               1
land grabbing court,salem                       1
Name: count, Length: 218, dtype: int64

In [41]:
# State impact
state_stats = df_model.groupby('state_code')['resolution_days'].median().round(0)
print(f"State median range: {state_stats.min()} to {state_stats.max()} days")

# District impact  
dist_stats = df_model.groupby('dist_code')['resolution_days'].median().round(0)
print(f"District median range: {dist_stats.min()} to {dist_stats.max()} days")

# Court impact
court_stats = df_model.groupby('court_no')['resolution_days'].median().round(0)
print(f"Court median range: {court_stats.min()} to {court_stats.max()} days")

State median range: 43.0 to 1316.0 days
District median range: 31.0 to 2599.0 days
Court median range: 180.0 to 2560.0 days


In [42]:
# Does judge_position have signal?
judge_stats = df_model.groupby('judge_position')['resolution_days'].median().round(0)
print(f"Judge position median range: {judge_stats.min()} to {judge_stats.max()} days")
print(f"Unique judge positions: {df_model['judge_position'].nunique()}")

# Does type_name have signal?
type_stats = df_model.groupby('type_name')['resolution_days'].median().round(0)
print(f"\nType name median range: {type_stats.min()} to {type_stats.max()} days")
print(f"Unique type names: {df_model['type_name'].nunique()}")

# Does act have signal?
act_stats = df_model.groupby('act')['resolution_days'].median().round(0)
print(f"\nAct median range: {act_stats.min()} to {act_stats.max()} days")
print(f"Unique acts: {df_model['act'].nunique()}")

Judge position median range: 2.0 to 3270.0 days
Unique judge positions: 218

Type name median range: 19.0 to 2513.0 days
Unique type names: 112

Act median range: 1.0 to 3263.0 days
Unique acts: 2986


In [43]:
# Check act frequency distribution
act_counts = df_model['act'].value_counts()
print(f"Acts with only 1 case: {(act_counts == 1).sum()}")
print(f"Acts with less than 10 cases: {(act_counts < 10).sum()}")
print(f"Acts with less than 100 cases: {(act_counts < 100).sum()}")
print(f"\nTop 20 acts cover how many cases:")
print(f"{act_counts.head(20).sum():,} out of {len(df_model):,} ({act_counts.head(20).sum()/len(df_model)*100:.1f}%)")

Acts with only 1 case: 854
Acts with less than 10 cases: 1864
Acts with less than 100 cases: 2552

Top 20 acts cover how many cases:
902,268 out of 1,291,086 (69.9%)


In [44]:
# Check top 50 and top 100
print(f"Top 50 acts cover: {act_counts.head(50).sum():,} ({act_counts.head(50).sum()/len(df_model)*100:.1f}%)")
print(f"Top 100 acts cover: {act_counts.head(100).sum():,} ({act_counts.head(100).sum()/len(df_model)*100:.1f}%)")

# Show top 20 act names so we understand what they are
act_key_peek = pd.read_csv(r"C:\Users\Karnaveer Singh\Justice_Hq\csv\keys\act_key.csv", nrows=5000)
print(act_key_peek.columns.tolist())

Top 50 acts cover: 1,044,291 (80.9%)
Top 100 acts cover: 1,135,886 (88.0%)
['act_s', 'count', 'act']


In [45]:
# Get top 50 act codes
top_50_acts = act_counts.head(50).index.tolist()

# Map to readable names
act_key_dedup = act_key_peek[['act', 'act_s']].drop_duplicates(subset='act')
top_50_names = act_key_dedup[act_key_dedup['act'].isin(top_50_acts)][['act','act_s']]
print(top_50_names.to_string())

         act                                          act_s
14      14.0                                              -
17      17.0                                            ---
19      19.0                                              .
2254  2254.0  A.P. COURT-FEES AND SUITS VALUATION ACT, 1956
2301  2301.0                                            ACT
2302  2302.0                                   ACT / U/Sec.
4069  4069.0                        CODE OF CIVIL PROCEDURE
4074  4074.0             CODE OF CIVIL PROCEDURE, 1908 (HB)
4161  4161.0                                            CPC
4650  4650.0                           Civil Procedure Code
4658  4658.0                     Civil Procedure Code, 1908
4689  4689.0                                CivilDefenceAct
4743  4743.0                        Code Of Civil Procedure
4747  4747.0                        Code of Civil Procedure
4750  4750.0                   Code of Civil Procedure 1908
4763  4763.0                           C

In [46]:
df_model.head()

,ddl_case_id,cino,state_code,dist_code,court_no,judge_position,type_name,act,section,date_of_filing,date_of_decision,resolution_days,resolution_bucket,filing_year,filing_quarter
0,01-03-02-207401000012010,MHJG040004222010,1,3,2,district and sessions court,4784.0,7276.0,1345422.0,2010-02-11,2011-03-30,412,1_six_to_24months,2010,1
1,01-03-02-207401000012011,MHJG040039232010,1,3,2,district and sessions court,4784.0,16320.0,635313.0,2010-12-24,2012-07-11,565,1_six_to_24months,2010,4
2,01-03-02-207401000022010,MHJG040006132010,1,3,2,district and sessions court,4784.0,16320.0,635313.0,2010-03-02,2012-06-25,846,2_over_2years,2010,1
3,01-03-02-207401000022011,MHJG040036672010,1,3,2,district and sessions court,4784.0,16320.0,643810.0,2010-12-04,2012-12-17,744,2_over_2years,2010,4
4,01-03-02-207401000032010,MHJG040016072010,1,3,2,district and sessions court,4784.0,16320.0,678419.0,2010-06-10,2011-01-03,207,1_six_to_24months,2010,2


In [47]:
df_model["judge_position"].value_counts()

judge_position
munsiff first class court                  150784
principal civil judge                      127553
civil judge senior division                125092
district and sessions court                112132
judicial magistrate court                  106555
                                            ...  
magistrate court miryalaguda                    1
co-operative appellate court aurangabad         1
gram nyayalaya                                  1
magistrate courts                               1
land grabbing court,salem                       1
Name: count, Length: 218, dtype: int64

In [48]:
import pandas as pd

act_key = pd.read_csv("csv/keys/act_key.csv")
section_key = pd.read_csv("csv/keys/section_key.csv")
type_name_key = pd.read_csv("csv/keys/type_name_key.csv")

act_key.head(), section_key.head(), type_name_key.head()

# Build compact lookup tables for the codes actually used in df_model
type_name_lookup = (
    df_model[['type_name']]
    .dropna()
    .drop_duplicates()
    .merge(type_name_key[['type_name', 'type_name_s']].drop_duplicates(), on='type_name', how='left')
)
section_lookup = (
    df_model[['section']]
    .dropna()
    .drop_duplicates()
    .merge(section_key[['section', 'section_s']].drop_duplicates(), on='section', how='left')
)
act_lookup = (
    df_model[['act']]
    .dropna()
    .drop_duplicates()
    .merge(act_key[['act', 'act_s']].drop_duplicates(), on='act', how='left')
)

# Save the mapping files
type_name_lookup.to_csv('csv/keys/type_name_lookup.csv', index=False)
section_lookup.to_csv('csv/keys/section_lookup.csv', index=False)
act_lookup.to_csv('csv/keys/act_lookup.csv', index=False)

print(type_name_lookup.head())
print(section_lookup.head())
print(act_lookup.head())

   type_name                               type_name_s
0     4784.0                       spl. marriage petn.
1     4784.0                              reg. misc 74
2     4784.0  partion suit / title partion suit(civil)
3     4784.0                                 obscenity
4     4784.0                         o.s. (decl.) suit
     section section_s
0  1345422.0         9
1   635313.0        27
2   643810.0      271d
3   678419.0        28
4   645440.0      2738
       act                         act_s
0   7276.0            Hindu Marriage Act
1  16320.0          Special Marriage Act
2   4663.0         Civil Procedure codes
3  10564.0             Motor Vehicle Act
4  13283.0  Prevention of Corruption Act


In [49]:
act_clean = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\act_clean.csv')
type_clean = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\type_clean.csv')

# Drop null act name
act_clean = act_clean.dropna(subset=['act_canonical'])

# Keep only commercial and civil acts for your use case
act_commercial = act_clean[act_clean['category'].isin(['Commercial', 'Civil'])]

print(f"Commercial + Civil acts: {len(act_commercial)}")
print(f"Type codes total: {len(type_clean)}")
print(f"Commercial type codes: {type_clean['is_commercial'].sum()}")

Commercial + Civil acts: 816
Type codes total: 112
Commercial type codes: 78


In [50]:
# Reset — go back to before your last filter
# You need to reload df_model from your saved state
# If you still have it in memory before the filter, run:

before_rows = len(df_model)

# Type name filter only — this is your cleanest signal
allowed_type_codes = set(type_clean.loc[type_clean['is_commercial'], 'type_name'].dropna())

df_model = df_model[df_model['type_name'].isin(allowed_type_codes)].copy()

print(f"Rows before: {before_rows:,}")
print(f"Rows after: {len(df_model):,}")
print(f"Dropped: {before_rows - len(df_model):,}")
print(f"\nTarget distribution:")
print(df_model['resolution_bucket'].value_counts().sort_index())

Rows before: 1,291,086
Rows after: 493,776
Dropped: 797,310

Target distribution:
resolution_bucket
0_under_6months      158188
1_six_to_24months    185906
2_over_2years        149682
Name: count, dtype: int64


In [52]:
df_model.head()

,ddl_case_id,cino,state_code,dist_code,court_no,judge_position,type_name,act,section,date_of_filing,date_of_decision,resolution_days,resolution_bucket,filing_year,filing_quarter
0,01-03-02-207401000012010,MHJG040004222010,1,3,2,district and sessions court,4784.0,7276.0,1345422.0,2010-02-11,2011-03-30,412,1_six_to_24months,2010,1
1,01-03-02-207401000012011,MHJG040039232010,1,3,2,district and sessions court,4784.0,16320.0,635313.0,2010-12-24,2012-07-11,565,1_six_to_24months,2010,4
2,01-03-02-207401000022010,MHJG040006132010,1,3,2,district and sessions court,4784.0,16320.0,635313.0,2010-03-02,2012-06-25,846,2_over_2years,2010,1
3,01-03-02-207401000022011,MHJG040036672010,1,3,2,district and sessions court,4784.0,16320.0,643810.0,2010-12-04,2012-12-17,744,2_over_2years,2010,4
4,01-03-02-207401000032010,MHJG040016072010,1,3,2,district and sessions court,4784.0,16320.0,678419.0,2010-06-10,2011-01-03,207,1_six_to_24months,2010,2


In [53]:
features = [
    'ddl_case_id', 'cino',
    'state_code', 'dist_code', 'court_no',
    'judge_position', 'type_name',
    'filing_year', 'filing_quarter',
    'resolution_days', 'resolution_bucket'
]

df_final = df_model[features].copy()

print(f"Final shape: {df_final.shape}")
print(f"Missing values:\n{df_final.isnull().sum()}")

df_final.to_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\df_model.csv', index=False)
print("\nSaved — Day 2 complete.")

Final shape: (493776, 11)
Missing values:
ddl_case_id          0
cino                 0
state_code           0
dist_code            0
court_no             0
judge_position       0
type_name            0
filing_year          0
filing_quarter       0
resolution_days      0
resolution_bucket    0
dtype: int64

Saved — Day 2 complete.


## Day 2 Summary — Feature Engineering Complete

Final modeling dataset: 493,776 rows
Features: state_code, dist_code, court_no, judge_position, 
          type_name, filing_year, filing_quarter
Target: resolution_bucket (3 classes)

Target distribution:
- Under 6 months:  32.1% (158k cases)
- 6 to 24 months:  37.7% (186k cases)  
- Over 2 years:    30.3% (150k cases)

Nearly balanced — no class weighting needed.

Dropped: section (110k unique values, no signal without act context)
Dropped: act (2986 unique values, messy naming across states)
Dropped: 2016-2018 filings (survivorship bias)
Kept: type_name filtered to 78 confirmed commercial codes